# Full-corpus LLM annotation

Section 1 splits the 6,501 cleaned reviews into CSV batches of `review_id,review`
to upload to ChatGPT. Section 2 reads the `review_id,topic_number` outputs back in
and merges them onto the cleaned dataset.

`review_id` is the row index of `data/stylecom_cleaned.csv`, which is the same
as the `doc_id` used in 2.7 and 2.8.

## 1. write the batches

In [ ]:
from pathlib import Path
import pandas as pd

BATCH_SIZE = 200

BATCH_DIR = Path("inputs_llm_annotation/batches")
RESULTS_PATH = Path("data/full_llm_annotation.csv")
FULL_DATASET_PATH = Path("data/stylecom_cleaned_annotated.csv")
BATCH_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv("data/stylecom_cleaned.csv")
df.insert(0, "review_id", range(len(df)))

print(f"{len(df)} reviews")
df.head()

6501 reviews


,review_id,year,season,designer,author,city,date,review
0,0,2000,Spring,Matt Nye,Armand Limnander,New York,17-Sep-99,Designer Matt Nye's sophomore show featured a ...
1,1,2000,Spring,Giorgio Armani,Armand Limnander,Milan,29-Sep-99,"Armani proposed a light, feminine silhouette f..."
2,2,2000,Spring,Eric Bergère,Armand Limnander,Paris,4-Oct-99,Broadway Garnier was the theme for Eric Berg'r...
3,3,2000,Spring,Céline,Armand Limnander,Paris,7-Oct-99,Getaway glamour was the theme for Celine's str...
4,4,2000,Spring,Byblos,Armand Limnander,Milan,27-Sep-99,Judo Jetson blends my favorite cartoon charact...


In [4]:
for start in range(0, len(df), BATCH_SIZE):
    batch = df.iloc[start:start + BATCH_SIZE]
    name = f"reviews_{start + 1:04d}_{min(start + BATCH_SIZE, len(df)):04d}.csv"
    batch[["review_id", "review"]].to_csv(BATCH_DIR / name, index=False)

print(f"{len(list(BATCH_DIR.glob('reviews_*.csv')))} batches written to {BATCH_DIR}/")

33 batches written to outputs_llm_annotation/batches/


### Setup

Make a ChatGPT project, let it have project-only memory, upload `data/topic_codebook.md`
then use a fresh chat per batch. Save each output to `outputs_llm_annotation/results/` under the same filename as the batch. Use the prompt from the Appendix of the thesis.



## 2. combine the results

In [ ]:
results = pd.read_csv(RESULTS_PATH)

results = results[["review_id", "topic_number"]].drop_duplicates(subset="review_id")
annotated = df.merge(results, on="review_id", how="left")

missing = annotated["topic_number"].isna().sum()
out_of_range = ~annotated["topic_number"].isin(range(10)) & annotated["topic_number"].notna()

annotated.to_csv(RESULTS_PATH, index=False)

print(f"{len(annotated) - missing}/{len(annotated)} reviews labelled")
print(f"missing: {missing}")
print(f"outside 0-9: {out_of_range.sum()}")

6501/6501 reviews labelled
missing: 0
outside 0-9: 0
